In [ ]:
import gymnasium as gym
import gymnasium_robotics

gym.register_envs(gymnasium_robotics)

env = gym.make(
    "FrankaKitchen-v1",
    tasks_to_complete=["microwave"],   # 或 ["bottom burner"] 等
    render_mode="rgb_array",
)

In [ ]:
import gymnasium as gym
import gymnasium_robotics
import numpy as np
import pickle
from pathlib import Path

gym.register_envs(gymnasium_robotics)


def make_env(task_name: str):
    env = gym.make(
        "FrankaKitchen-v1",
        tasks_to_complete=[task_name],
        render_mode="rgb_array",
    )
    return env


def get_expert_action(obs_dict, image):
    """
    你自己实现：
    - 键盘 teleop
    - 手柄输入
    - 旧 policy
    - scripted controller
    返回 shape=(9,) 的 action，范围建议 clip 到 [-1, 1]
    """
    raise NotImplementedError


def collect_one_episode(env, task_name: str, camera_name: str = "default"):
    obs, info = env.reset()

    # goal-aware obs: obs 是 dict，包含 observation / desired_goal / achieved_goal
    desired_goal = obs["desired_goal"]

    observations = []
    actions = []
    rewards = []
    images = []

    achieved_goals = []
    step_task_completions = []
    episode_task_completions = []
    tasks_to_complete = []

    terminated = False
    truncated = False

    while not (terminated or truncated):
        image = env.render()   # rgb_array
        action = get_expert_action(obs, image)
        action = np.asarray(action, dtype=np.float32)
        action = np.clip(action, -1.0, 1.0)

        observations.append(np.asarray(obs["observation"], dtype=np.float64))
        images.append(np.asarray(image, dtype=np.uint8))
        actions.append(action)
        achieved_goals.append(obs["achieved_goal"])

        obs, reward, terminated, truncated, info = env.step(action)

        rewards.append(float(reward))
        step_task_completions.append(info.get("step_task_completions", []))
        episode_task_completions.append(info.get("episode_task_completions", []))
        tasks_to_complete.append(info.get("tasks_to_complete", []))

    final_completed = episode_task_completions[-1] if len(episode_task_completions) > 0 else []
    success = task_name in final_completed or bool(terminated)

    traj = {
        "task_name": task_name,
        "camera_name": camera_name,

        "observations": np.asarray(observations),
        "actions": np.asarray(actions),
        "rewards": np.asarray(rewards),
        "images": np.asarray(images),

        "desired_goal": desired_goal,
        "achieved_goal": achieved_goals,

        "step_task_completions": step_task_completions,
        "episode_task_completions": episode_task_completions,
        "tasks_to_complete": tasks_to_complete,

        "terminated": bool(terminated),
        "truncated": bool(truncated),
        "success": bool(success),
        "length": len(actions),
    }
    return traj


def collect_demos(task_name: str, num_demos: int, save_path: str):
    env = make_env(task_name)
    demos = []

    for i in range(num_demos):
        print(f"[{task_name}] collecting demo {i+1}/{num_demos}")
        traj = collect_one_episode(env, task_name=task_name, camera_name="default")
        print(f"  success={traj['success']}, length={traj['length']}")
        demos.append(traj)

    env.close()

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "wb") as f:
        pickle.dump(demos, f)

    print(f"saved {len(demos)} demos to {save_path}")


if __name__ == "__main__":
    collect_demos(
        task_name="microwave",
        num_demos=20,
        save_path="./new_demos/microwave.pkl",
    )